# Agregação de Resultados de Benchmark

Coleta dados de energia (CodeCarbon) e performance HTTP de N rodadas de execução,
agrega por versão de framework e exporta dois CSVs:

- **`rounds.csv`** — uma linha por `(exc_n, framework, framework_version)`
- **`summary_by_version.csv`** — `mean`, `std` e `CV` entre as N rodadas por versão

### Estrutura de diretórios esperada
```
result_exc_N/
  results_<framework>_<version>/
    carbon/emissions.csv
    logs/{api,html,json}_max_requests.csv
```


In [1]:
import sys
!{sys.executable} -m pip install numpy pandas

## 1. Imports

In [4]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd


## 2. Configuração

Ajuste `ROOT_DIR` para o diretório que contém as pastas `results_exc_N/`.  
Os arquivos de saída serão criados no mesmo diretório.


In [5]:
ROOT_DIR    = Path("results_exc_all")  # pasta contendo todos os results_exc_N/
OUTPUT_CSV  = Path("rounds.csv")
SUMMARY_CSV = Path("summary_by_version.csv")


## 3. Constantes

In [6]:
EXC_RE         = re.compile(r"results_exc_(\d+)", re.IGNORECASE)
RESULTS_DIR_RE = re.compile(r"results_(.+)$",    re.IGNORECASE)

PERCENTILES  = [0.50, 0.75, 0.90, 0.95, 0.99]
PERC_LABELS  = ["p50", "p75", "p90", "p95", "p99"]
API_ENDPOINTS = ["api", "html", "upload"]

# Valores acumulados ao longo da rodada (última linha válida do emissions.csv)
EMISSION_COLS = [
    "duration",
    "emissions",        # kg CO2eq acumulado
    "cpu_energy",       # Energy used per CPU (kWh)
    "gpu_energy",       # Energy used per GPU (kWh)
    "ram_energy",       # Energy used per RAM (kWh)
    "energy_consumed",  # kWh total (principal métrica de energia)
]
# Valores de leitura instantânea (último snapshot com cpu_power > 0)
EMISSION_INSTANT_COLS = [
    "emissions_rate",   # kg CO2eq/s
    "cpu_power",        # Mean CPU power (W)
    "ram_power",        # Mean RAM power (W)
]


## 4. Parsing de diretórios

In [7]:
def parse_dir_name(dir_name: str) -> tuple[str, str]:
    """
    'results_fastapi_0_128_8' -> ('fastapi', '0.128.8')

    O primeiro segmento puramente numérico marca o início da versão.
    Tudo antes é o nome do framework.
    """
    m = RESULTS_DIR_RE.match(dir_name)
    if not m:
        return dir_name, ""
    parts = m.group(1).split("_")
    ver_start = next((i for i, p in enumerate(parts) if p.isdigit()), None)
    if ver_start is None:
        return "_".join(parts), ""
    return "_".join(parts[:ver_start]), ".".join(parts[ver_start:])


def parse_path(path: Path) -> tuple[int | None, str | None, str | None]:
    """Extrai (exc_n, framework, version) do caminho completo.

    Só considera partes do caminho que venham após um result_exc_N,
    evitando falso-positivo com o diretório pai results_exc_all.
    """
    exc_match = EXC_RE.search(str(path))
    exc_n = int(exc_match.group(1)) if exc_match else None
    framework, version = None, None
    found_exc = False
    for part in path.parts:
        if EXC_RE.search(part):
            found_exc = True
            continue
        if found_exc and RESULTS_DIR_RE.match(part):
            framework, version = parse_dir_name(part)
            break
    return exc_n, framework, version


def find_experiment_dirs(root: Path) -> list[Path]:
    """Lista todos os results_exc_N/results_<framework_version>/ dentro de results_exc_all."""
    dirs = []
    for exc_dir in sorted(root.glob("results_exc_*"),
                          key=lambda p: int(EXC_RE.search(p.name).group(1))):
        if not exc_dir.is_dir():
            continue
        for ver_dir in sorted(exc_dir.glob("results_*")):
            if ver_dir.is_dir():
                dirs.append(ver_dir)
    return dirs


## 5. Leitura de dados por rodada

In [8]:
def read_emissions(csv_path: Path) -> dict:
    """
    Retorna métricas de energia da rodada a partir do emissions.csv.
    Descarta linhas com cpu_power == 0.0 (CodeCarbon encerrou a leitura).

    Métricas acumuladas: última linha válida.
    Métricas instantâneas (W): calculadas sobre todas as linhas válidas.
    """
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            return {}

        if "cpu_power" in df.columns:
            valid = df[df["cpu_power"] > 0.0]
            if valid.empty:
                print(f"  [WARN] Todas as linhas com cpu_power=0 em {csv_path} — usando última linha mesmo assim.")
                valid = df
            elif len(valid) < len(df):
                print(f"  [INFO] {len(df) - len(valid)} linha(s) com cpu_power=0 descartada(s) "
                      f"em {csv_path.parent.parent.name}")
        else:
            valid = df

        last = valid.iloc[-1]
        result = {}

        # Valores acumulados (última linha válida)
        for col in EMISSION_COLS + EMISSION_INSTANT_COLS:
            if col in last.index:
                result[f"emission_{col}"] = last[col]

        # Potência total por snapshot (CPU + RAM) em Watts
        if "cpu_power" in valid.columns and "ram_power" in valid.columns:
            total_power = valid["cpu_power"] + valid["ram_power"]
            result["emission_power_mean"] = total_power.mean()    # MnW
            result["emission_power_median"] = total_power.median() # MdW

        # CO2eq estimado por ano extrapolando a taxa média (kg/s → kg/ano)
        if "emissions_rate" in valid.columns:
            result["emission_co2eq_year"] = valid["emissions_rate"].mean() * 60 * 60 * 24 * 365

        for meta in ["python_version", "cpu_model", "cpu_count", "ram_total_size",
                     "country_name", "region", "on_cloud"]:
            if meta in last.index:
                result[meta] = last[meta]
        return result
    except Exception as e:
        print(f"  [WARN] Falha ao ler {csv_path}: {e}")
        return {}

def read_api_log(csv_path: Path, endpoint: str) -> dict:
    """
    Calcula estatísticas de response-time de uma rodada para um endpoint.
    Considera apenas requisições com status 2xx.
    """
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            return {}

        ok = df[df["status-code"].between(200, 299)].copy()
        total, success = len(df), len(ok)

        errors = total - success
        result = {
            f"{endpoint}_requests_total":   total,
            f"{endpoint}_requests_success": success,
            f"{endpoint}_requests_error":   errors,
            f"{endpoint}_success_rate":     round(success / total, 10) if total else 0,
        }
        if ok.empty:
            return result

        rt = ok["response-time"]
        result[f"{endpoint}_rt_mean"] = rt.mean()
        result[f"{endpoint}_rt_std"]  = rt.std()
        result[f"{endpoint}_rt_min"]  = rt.min()
        result[f"{endpoint}_rt_max"]  = rt.max()
        for p, label in zip(PERCENTILES, PERC_LABELS):
            result[f"{endpoint}_rt_{label}"] = rt.quantile(p)

        if "offset" in ok.columns:
            window = ok["offset"].max() - ok["offset"].min()
            if window > 0:
                result[f"{endpoint}_throughput_rps"] = success / window

        for col in ["Response-delay", "Response-read", "Request-write"]:
            if col in ok.columns:
                key = col.lower().replace("-", "_")
                result[f"{endpoint}_{key}_mean"] = ok[col].mean()
                result[f"{endpoint}_{key}_p95"]  = ok[col].quantile(0.95)

        return result
    except Exception as e:
        print(f"  [WARN] Falha ao ler {csv_path}: {e}")
        return {}


## 6. Coleta — uma linha por rodada

In [9]:
def collect_rounds(root: Path) -> pd.DataFrame:
    """Varre todos os result_exc_N e retorna um DataFrame com uma linha por rodada."""
    ver_dirs = find_experiment_dirs(root)
    if not ver_dirs:
        raise FileNotFoundError(f"Nenhum diretório results_exc_all/result_exc_N/results_* encontrado em: {root}")

    print(f"Encontradas {len(ver_dirs)} combinação(ões) rodada×versão.")
    rows = []

    for ver_dir in ver_dirs:
        exc_n, framework, version = parse_path(ver_dir)
        #print(f"  exc={exc_n:>3}  {framework} {version}")

        row: dict = {
            "exc_n":             exc_n,
            "framework":         framework,
            "framework_version": version,
        }

        emissions_csv = ver_dir / "carbon" / "emissions.csv"
        if emissions_csv.exists():
            row.update(read_emissions(emissions_csv))
        else:
            print(f"    [WARN] emissions.csv não encontrado em {ver_dir}")

        logs_dir = ver_dir / "logs"
        for endpoint in API_ENDPOINTS:
            log_csv = logs_dir / f"{endpoint}_max_requests.csv"
            if log_csv.exists():
                row.update(read_api_log(log_csv, endpoint))
            else:
                print(f"    [INFO] {endpoint}_max_requests.csv ausente")

        rows.append(row)

    df = (pd.DataFrame(rows)
            .sort_values(["framework", "framework_version", "exc_n"])
            .reset_index(drop=True))

    # Métricas derivadas: energia e CO2 por requisição (por endpoint)
    total_success = pd.Series(0, index=df.index)
    for endpoint in API_ENDPOINTS:
        sc = f"{endpoint}_requests_success"
        if sc in df.columns:
            total_success = total_success + df[sc].fillna(0)

    if "emission_energy_consumed" in df.columns:
        df["emission_energy_per_request"] = (
            df["emission_energy_consumed"] / total_success.replace(0, pd.NA)
        )
    if "emission_emissions" in df.columns:
        df["emission_co2_per_request"] = (
            df["emission_emissions"] / total_success.replace(0, pd.NA)
        )

    return df


### Executar coleta

In [10]:
rounds_df = collect_rounds(ROOT_DIR)
rounds_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nDados brutos salvos em: {OUTPUT_CSV}  ({len(rounds_df)} rodadas)")
rounds_df.head()


Encontradas 13000 combinação(ões) rodada×versão.
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_10_1
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_10_10
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_10_11
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_11_12
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_11_13
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_11_18
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_11_5
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_11_8
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_12_1
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_12_14
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_12_4
  [INFO] 3 linha(s) com cpu_power=0 descartada(s) em results_aiohttp_3_12_6
  [INFO] 3 linha(s) com cpu_power

,exc_n,framework,framework_version,emission_duration,emission_emissions,emission_cpu_energy,emission_gpu_energy,emission_ram_energy,emission_energy_consumed,emission_emissions_rate,...,upload_rt_p99,upload_throughput_rps,upload_response_delay_mean,upload_response_delay_p95,upload_response_read_mean,upload_response_read_p95,upload_request_write_mean,upload_request_write_p95,emission_energy_per_request,emission_co2_per_request
0,1,aiohttp,3.10.0,13.399811,0.000018,0.000005,0.0,0.000022,0.000027,0.000001,...,0.0395,2012.173002,0.031685,0.033600,0.000035,0.0001,0.000019,0.0001,0.0,0.0
1,2,aiohttp,3.10.0,11.218805,0.000015,0.000004,0.0,0.000019,0.000022,0.000001,...,0.0371,2565.064358,0.024863,0.031100,0.000024,0.0001,0.000011,0.0000,0.0,0.0
2,3,aiohttp,3.10.0,11.007722,0.000015,0.000003,0.0,0.000019,0.000022,0.000001,...,0.0373,2607.401217,0.024367,0.031100,0.000020,0.0001,0.000012,0.0000,0.0,0.0
3,4,aiohttp,3.10.0,14.546232,0.000018,0.000006,0.0,0.000021,0.000027,0.000001,...,0.0453,1796.879218,0.035563,0.039785,0.000050,0.0001,0.000019,0.0001,0.0,0.0
4,5,aiohttp,3.10.0,12.513453,0.000017,0.000005,0.0,0.000021,0.000025,0.000001,...,0.0393,2116.150911,0.030118,0.032900,0.000030,0.0001,0.000010,0.0001,0.0,0.0


In [12]:
def filter_invalid_versions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove do DataFrame todas as linhas cujo (framework, framework_version)
    tenha ao menos uma rodada com respostas fora do intervalo 2xx em qualquer
    endpoint (api, html, upload).

    Imprime quais versões foram removidas e o motivo.
    """
    error_cols = [c for c in df.columns if c.endswith("_requests_error")]
    if not error_cols:
        print("[INFO] Nenhuma coluna de erros encontrada — nada filtrado.")
        return df

    group_cols = ["framework", "framework_version"]

    # Versões com ao menos 1 erro em qualquer endpoint em qualquer rodada
    has_error = df[error_cols].gt(0).any(axis=1)
    bad_versions = df.loc[has_error, group_cols].drop_duplicates()

    if bad_versions.empty:
        print("✓ Nenhuma versão com erros de status HTTP encontrada.")
        return df

    print(f"[!] {len(bad_versions)} versão(ões) removida(s) por erros HTTP fora de 2xx:\n")
    for _, row in bad_versions.iterrows():
        mask = (df["framework"] == row["framework"]) & (df["framework_version"] == row["framework_version"])
        rounds_with_error = df.loc[mask & has_error]
        for _, r in rounds_with_error.iterrows():
            for col in error_cols:
                if r[col] > 0:
                    endpoint = col.replace("_requests_error", "")
                    print(f"  {row['framework']} {row['framework_version']} "
                          f"(exc={int(r['exc_n'])}) — {endpoint}: {int(r[col])} erro(s)")
        print()

    # Filtra
    bad_idx = df[group_cols].apply(tuple, axis=1).isin(bad_versions.apply(tuple, axis=1))
    return df[~bad_idx].reset_index(drop=True)


In [13]:
rounds_df = filter_invalid_versions(rounds_df)
print(f"Rodadas restantes para agregação: {len(rounds_df)}")
rounds_df.to_csv("rounds_filtered.csv", index=False)
rounds_df.head()

[!] 45 versão(ões) removida(s) por erros HTTP fora de 2xx:

  baize 0.10.0 (exc=1) — upload: 9984 erro(s)
  baize 0.10.0 (exc=2) — upload: 9984 erro(s)
  baize 0.10.0 (exc=3) — upload: 9984 erro(s)
  baize 0.10.0 (exc=4) — upload: 9984 erro(s)
  baize 0.10.0 (exc=5) — upload: 9984 erro(s)
  baize 0.10.0 (exc=6) — upload: 9984 erro(s)
  baize 0.10.0 (exc=7) — upload: 9984 erro(s)
  baize 0.10.0 (exc=8) — upload: 9984 erro(s)
  baize 0.10.0 (exc=9) — upload: 9984 erro(s)
  baize 0.10.0 (exc=10) — upload: 9984 erro(s)
  baize 0.10.0 (exc=11) — upload: 9984 erro(s)
  baize 0.10.0 (exc=12) — upload: 9984 erro(s)
  baize 0.10.0 (exc=13) — upload: 9984 erro(s)
  baize 0.10.0 (exc=14) — upload: 9984 erro(s)
  baize 0.10.0 (exc=15) — upload: 9984 erro(s)
  baize 0.10.0 (exc=16) — upload: 9984 erro(s)
  baize 0.10.0 (exc=17) — upload: 9984 erro(s)
  baize 0.10.0 (exc=18) — upload: 9984 erro(s)
  baize 0.10.0 (exc=19) — upload: 9984 erro(s)
  baize 0.10.0 (exc=20) — upload: 9984 erro(s)

  baize 

,exc_n,framework,framework_version,emission_duration,emission_emissions,emission_cpu_energy,emission_gpu_energy,emission_ram_energy,emission_energy_consumed,emission_emissions_rate,...,upload_rt_p99,upload_throughput_rps,upload_response_delay_mean,upload_response_delay_p95,upload_response_read_mean,upload_response_read_p95,upload_request_write_mean,upload_request_write_p95,emission_energy_per_request,emission_co2_per_request
0,1,aiohttp,3.10.0,13.399811,0.000018,0.000005,0.0,0.000022,0.000027,0.000001,...,0.0395,2012.173002,0.031685,0.033600,0.000035,0.0001,0.000019,0.0001,0.0,0.0
1,2,aiohttp,3.10.0,11.218805,0.000015,0.000004,0.0,0.000019,0.000022,0.000001,...,0.0371,2565.064358,0.024863,0.031100,0.000024,0.0001,0.000011,0.0000,0.0,0.0
2,3,aiohttp,3.10.0,11.007722,0.000015,0.000003,0.0,0.000019,0.000022,0.000001,...,0.0373,2607.401217,0.024367,0.031100,0.000020,0.0001,0.000012,0.0000,0.0,0.0
3,4,aiohttp,3.10.0,14.546232,0.000018,0.000006,0.0,0.000021,0.000027,0.000001,...,0.0453,1796.879218,0.035563,0.039785,0.000050,0.0001,0.000019,0.0001,0.0,0.0
4,5,aiohttp,3.10.0,12.513453,0.000017,0.000005,0.0,0.000021,0.000025,0.000001,...,0.0393,2116.150911,0.030118,0.032900,0.000030,0.0001,0.000010,0.0001,0.0,0.0


## 7. Sumarização por versão

In [14]:
def summarize_by_version(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada (framework, framework_version), agrega as N rodadas calculando
    mean, std e CV (coeficiente de variação) das métricas numéricas.
    Adiciona n_rounds — número de rodadas encontradas.
    """
    group_cols   = ["framework", "framework_version"]
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in ("exc_n",)]

    if not numeric_cols:
        print("[WARN] Sem colunas numéricas para agregar.")
        return df.groupby(group_cols)["exc_n"].nunique().reset_index(name="n_rounds")

    agg = df.groupby(group_cols)[numeric_cols].agg(["mean", "std"])
    agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]
    agg = agg.reset_index()

    # CV = std / mean (dispersão relativa entre rodadas)
    for col in numeric_cols:
        mean_c, std_c = f"{col}_mean", f"{col}_std"
        if mean_c in agg.columns and std_c in agg.columns:
            agg[f"{col}_cv"] = (agg[std_c] / agg[mean_c].replace(0, np.nan)).round(10)

    n_rounds = df.groupby(group_cols)["exc_n"].nunique().reset_index(name="n_rounds")
    agg = agg.merge(n_rounds, on=group_cols)

    front = group_cols + ["n_rounds"]
    agg = agg[front + [c for c in agg.columns if c not in front]]
    return agg.sort_values(group_cols).reset_index(drop=True)


### Executar sumarização

In [15]:
summary_df = summarize_by_version(rounds_df)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Sumário salvo em: {SUMMARY_CSV}  ({len(summary_df)} versão(ões))")
summary_df


Sumário salvo em: summary_by_version.csv  (605 versão(ões))


,framework,framework_version,n_rounds,emission_duration_mean,emission_duration_std,emission_emissions_mean,emission_emissions_std,emission_cpu_energy_mean,emission_cpu_energy_std,emission_gpu_energy_mean,...,upload_rt_p90_cv,upload_rt_p95_cv,upload_rt_p99_cv,upload_throughput_rps_cv,upload_response_delay_mean_cv,upload_response_delay_p95_cv,upload_response_read_mean_cv,upload_response_read_p95_cv,upload_request_write_mean_cv,upload_request_write_p95_cv
0,aiohttp,3.10.0,20,12.360789,1.236847,0.000016,0.000002,0.000004,9.084552e-07,0.0,...,0.139390,0.165957,0.148214,0.197823,0.161521,0.166689,0.362227,0.235376,0.299115,0.837708
1,aiohttp,3.10.1,20,12.361830,1.442127,0.000016,0.000002,0.000004,1.145865e-06,0.0,...,0.172846,0.196775,0.233068,0.235536,0.195908,0.196244,0.457870,0.235376,0.495669,0.837708
2,aiohttp,3.10.10,20,12.486411,1.016775,0.000016,0.000002,0.000005,8.653703e-07,0.0,...,0.090664,0.129610,0.106973,0.117206,0.119739,0.129502,0.326727,0.000000,0.310182,0.837708
3,aiohttp,3.10.11,20,12.021799,1.290343,0.000016,0.000002,0.000004,9.457530e-07,0.0,...,0.131979,0.151267,0.139172,0.216390,0.177345,0.150178,0.379197,0.235376,0.414708,0.928032
4,aiohttp,3.10.2,20,12.393283,1.184622,0.000016,0.000002,0.000005,9.359084e-07,0.0,...,0.148480,0.167917,0.155535,0.199787,0.164541,0.166069,0.389313,0.235376,0.327659,1.025978
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,tornado,6.5,20,13.646866,0.692832,0.000017,0.000001,0.000005,4.731711e-07,0.0,...,0.080492,0.094352,0.128450,0.113085,0.112709,0.092727,0.278391,0.000000,0.380299,4.472136
601,tornado,6.5.1,20,13.488772,0.748875,0.000018,0.000001,0.000005,5.002200e-07,0.0,...,0.082902,0.114255,0.120971,0.117315,0.117621,0.113615,0.306137,0.000000,0.336733,NaN
602,tornado,6.5.2,20,13.555196,0.729724,0.000018,0.000001,0.000005,6.201073e-07,0.0,...,0.079285,0.106581,0.128999,0.111030,0.113996,0.107648,0.254238,0.000000,0.462550,4.472136
603,tornado,6.5.3,20,13.373394,0.794632,0.000017,0.000001,0.000005,4.612539e-07,0.0,...,0.075091,0.081298,0.101817,0.118975,0.116132,0.080641,0.298720,0.235376,0.338354,NaN


## 8. Preview dos resultados

In [16]:
ecol_m = "emission_energy_consumed_mean"
ecol_s = "emission_energy_consumed_std"

if ecol_m in summary_df.columns:
    display(summary_df[["framework", "framework_version", "n_rounds", ecol_m, ecol_s]]
              .style.format({ecol_m: "{:.2e}", ecol_s: "{:.2e}"})
              .set_caption("Energia consumida por versão (kWh)"))


,framework,framework_version,n_rounds,emission_energy_consumed_mean,emission_energy_consumed_std
0,aiohttp,3.10.0,20,2.40e-05,2.46e-06
1,aiohttp,3.10.1,20,2.36e-05,3.14e-06
2,aiohttp,3.10.10,20,2.43e-05,2.45e-06
3,aiohttp,3.10.11,20,2.34e-05,2.78e-06
4,aiohttp,3.10.2,20,2.42e-05,3.15e-06
5,aiohttp,3.10.3,20,2.43e-05,3.26e-06
6,aiohttp,3.10.4,20,2.39e-05,3.15e-06
7,aiohttp,3.10.5,20,2.37e-05,3.25e-06
8,aiohttp,3.10.6,20,2.32e-05,2.56e-06
9,aiohttp,3.10.7,20,2.33e-05,2.50e-06


In [22]:
tcol_m = "api_throughput_rps_mean"
tcol_s = "api_throughput_rps_std"

if tcol_m in summary_df.columns:
    display(summary_df[["framework", "framework_version", "n_rounds", tcol_m, tcol_s]]
              .style.format({tcol_m: "{:.2f}", tcol_s: "{:.2f}"})
              .set_caption("Throughput API (req/s)"))


,framework,framework_version,n_rounds,api_throughput_rps_mean,api_throughput_rps_std
0,aiohttp,3.10.0,20,5872.19,1181.57
1,aiohttp,3.10.1,20,6228.17,1549.13
2,aiohttp,3.10.10,20,5950.41,1489.36
3,aiohttp,3.10.11,20,6255.89,1615.31
4,aiohttp,3.10.2,20,5867.72,1278.98
5,aiohttp,3.10.3,20,6079.04,1585.59
6,aiohttp,3.10.4,20,5853.40,1402.62
7,aiohttp,3.10.5,20,5701.07,1324.54
8,aiohttp,3.10.6,20,6388.04,1621.88
9,aiohttp,3.10.7,20,6295.56,1632.53
